# Task 02 — Neural CTR Prediction
**Models:** LR baseline, MLP-A, MLP-B, MLP-C, embedding-dim variants, DCN  
**Data:** Criteo-style dataset in `../../dataset_2_3/`  
**Pipeline:** `common/ctr_data.py`, `common/metrics.py`, `common/utils.py`  
**Logic:** `02_NeuralCTR/models.py`, `02_NeuralCTR/train.py`

## How to rerun
```bash
pip install -r RecommendationTasks/requirements.txt
# Dataset should be at: Mandatory_Tasks/Recommender_Systems/dataset_2_3/
jupyter nbconvert --to notebook --execute neural_ctr.ipynb
```

In [ ]:
# Cell 1 — Environment check
import sys, platform
import torch
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}" + (f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Seed    : 42")

In [ ]:
# Cell 2 — Imports and paths
import pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Add common/ to path
ROOT = pathlib.Path('..').resolve()           # RecommendationTasks/
sys.path.insert(0, str(ROOT))

from common.utils import set_seed, get_device, count_params
from common.ctr_data import load_ctr_data
from common.metrics import compute_metrics, find_best_f1_threshold, plot_reliability_diagram
from models import LogisticRegression, MLPA, MLPB, MLPC, DCN, BaseMLP
from train import train_model, eval_on_test, save_results

set_seed(42)
DEVICE = get_device()

DATA_DIR   = pathlib.Path('../../dataset_2_3').resolve()
SPLIT_PATH = ROOT / 'common' / 'split.npz'
RESULTS_DIR = pathlib.Path('results')
FIGURES_DIR = pathlib.Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Data dir : {DATA_DIR}")
print(f"Split    : {SPLIT_PATH}")

In [ ]:
# Cell 3 — Load data  (runs once; reuses split.npz on subsequent runs)
train_loader, val_loader, test_loader, NUM_NUMERIC, CAT_CARDS = load_ctr_data(
    data_dir=DATA_DIR,
    split_save_path=SPLIT_PATH,
    batch_size=2048,
    min_freq=10,
    seed=42,
)

print(f"Numeric features : {NUM_NUMERIC}")
print(f"Categorical fields: {len(CAT_CARDS)}")
print(f"Cardinalities (first 5): {CAT_CARDS[:5]}  ...  (last 5): {CAT_CARDS[-5:]}")
print(f"Train batches    : {len(train_loader)} (batch=2048)")
print(f"Val   batches    : {len(val_loader)}")
print(f"Test  batches    : {len(test_loader)}")

In [ ]:
# Cell 4 — Helper: run one experiment and save results
def run_experiment(name, model, tag=None):
    tag = tag or name
    print(f"\n{'='*55}")
    print(f"  Training: {name}  ({count_params(model):,} params)")
    print(f"{'='*55}")

    set_seed(42)  # reset per model so comparison is fair
    run_info = train_model(
        model, train_loader, val_loader,
        lr=1e-3, weight_decay=1e-5,
        max_epochs=10, patience=2,
        device=DEVICE,
    )

    result = {
        "model": name,
        "n_params": run_info["n_params"],
        "best_val_loss": run_info["best_val_loss"],
        "best_epoch": run_info["best_epoch"],
        "history": run_info["history"],
        "peak_memory_mb": run_info["peak_memory_mb"],
        "best_threshold": run_info["best_threshold"],
    }
    save_results(result, RESULTS_DIR / f"{tag}.json")
    return model, run_info

all_results = {}   # name -> (model, run_info)

In [ ]:
# Cell 5 — LR baseline
lr_model = LogisticRegression(NUM_NUMERIC, CAT_CARDS)
all_results['LR'] = run_experiment('LR', lr_model, tag='lr_baseline')

In [ ]:
# Cell 6 — MLP-A  (embed=16, 1 hidden layer 128, no dropout)
mlpa = MLPA(NUM_NUMERIC, CAT_CARDS, embed_dim=16)
all_results['MLP-A'] = run_experiment('MLP-A', mlpa, tag='mlp_a')

In [ ]:
# Cell 7 — MLP-B  (embed=16, 2 hidden 256, dropout=0.2)
mlpb = MLPB(NUM_NUMERIC, CAT_CARDS, embed_dim=16)
all_results['MLP-B'] = run_experiment('MLP-B', mlpb, tag='mlp_b')

In [ ]:
# Cell 8 — MLP-C  (embed=16, 3 hidden 512, dropout=0.3)
mlpc = MLPC(NUM_NUMERIC, CAT_CARDS, embed_dim=16)
all_results['MLP-C'] = run_experiment('MLP-C', mlpc, tag='mlp_c')

In [ ]:
# Cell 9 — Embedding-dim variants on best MLP (use MLP-B as representative)
mlpb_d8 = MLPB(NUM_NUMERIC, CAT_CARDS, embed_dim=8)
all_results['MLP-B-d8'] = run_experiment('MLP-B (dim=8)', mlpb_d8, tag='mlp_b_d8')

mlpb_d32 = MLPB(NUM_NUMERIC, CAT_CARDS, embed_dim=32)
all_results['MLP-B-d32'] = run_experiment('MLP-B (dim=32)', mlpb_d32, tag='mlp_b_d32')

In [ ]:
# Cell 10 — DCN (3 cross layers + deep MLP [256,256])
dcn = DCN(NUM_NUMERIC, CAT_CARDS, embed_dim=16, mlp_hidden=[256, 256], mlp_dropout=0.2, cross_layers=3)
all_results['DCN'] = run_experiment('DCN', dcn, tag='dcn')

In [ ]:
# Cell 11 — Training loss curves for all models
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
axes = axes.flatten()

plot_names = ['LR', 'MLP-A', 'MLP-B', 'MLP-C', 'DCN', 'MLP-B-d8']

for ax, name in zip(axes, plot_names):
    if name not in all_results:
        ax.set_visible(False)
        continue
    _, info = all_results[name]
    hist = info['history']
    epochs = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs, hist['train_loss'], 'o-', label='Train loss', color='steelblue')
    ax.plot(epochs, hist['val_loss'],   's--', label='Val loss',   color='crimson')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Train vs Val Loss — All Neural CTR Models', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/loss_curves.png')

In [ ]:
# Cell 12 — Pick best model by val log loss, then evaluate ONCE on test
best_name = min(
    [n for n in all_results],
    key=lambda n: all_results[n][1]['best_val_loss']
)
best_model, best_info = all_results[best_name]
best_threshold = best_info['best_threshold']
print(f"Best model by val log loss: {best_name}")
print(f"  best_val_loss = {best_info['best_val_loss']:.4f}  at epoch {best_info['best_epoch']}")
print(f"  F1-optimal threshold (from val): {best_threshold:.3f}")

test_metrics = eval_on_test(best_model, test_loader, threshold=best_threshold, device=DEVICE)
print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")
save_results(
    {"model": best_name, "threshold": best_threshold, **test_metrics},
    RESULTS_DIR / 'best_model_test.json'
)

In [ ]:
# Cell 13 — Full metrics table (val) for all models
import pandas as pd
from common.metrics import find_best_f1_threshold
import torch.nn as nn

rows = []
criterion = nn.BCEWithLogitsLoss()

for name, (model, info) in all_results.items():
    # Re-evaluate on val to get probabilities (no test leakage)
    from train import evaluate
    val_loss, y_true_val, y_prob_val = evaluate(model, val_loader, criterion, DEVICE)
    thresh = info['best_threshold'] or find_best_f1_threshold(y_true_val, y_prob_val)
    m = compute_metrics(y_true_val, y_prob_val, threshold=thresh)

    rows.append({
        'Model': name,
        'ROC-AUC': round(m['roc_auc'], 4),
        'PR-AUC': round(m['pr_auc'], 4),
        'Log Loss': round(m['log_loss'], 4),
        'Accuracy': round(m['accuracy_0.5'], 4),
        'F1': round(m['f1'], 4),
        'Precision': round(m['precision'], 4),
        'Recall': round(m['recall'], 4),
        'ECE': round(m['ece'], 4),
        'Threshold': round(thresh, 3),
        'Params': info['n_params'],
        'Peak Mem (MB)': info['peak_memory_mb'],
        'Best Epoch': info['best_epoch'],
    })

df = pd.DataFrame(rows).sort_values('Log Loss')
print(df.to_string(index=False))
df.to_csv(RESULTS_DIR / 'summary_table.csv', index=False)
print('\nSaved → results/summary_table.csv')

In [ ]:
# Cell 14 — Reliability diagram for best model (val set)
_, best_val_probs = evaluate(best_model, val_loader, criterion, DEVICE)[1:]
_, y_true_val, y_prob_val = evaluate(best_model, val_loader, criterion, DEVICE)

plot_reliability_diagram(
    y_true_val, y_prob_val,
    model_name=best_name,
    save_path=FIGURES_DIR / 'reliability_diagram.png'
)
print('Saved → figures/reliability_diagram.png')

In [ ]:
# Cell 15 — ROC and PR curves for top 3 models by val log loss
from sklearn.metrics import roc_curve, precision_recall_curve

top3 = df['Model'].head(3).tolist()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for name in top3:
    model, info = all_results[name]
    _, y_true_v, y_prob_v = evaluate(model, val_loader, criterion, DEVICE)

    fpr, tpr, _ = roc_curve(y_true_v, y_prob_v)
    prec, rec, _ = precision_recall_curve(y_true_v, y_prob_v)

    auc_val = compute_metrics(y_true_v, y_prob_v)['roc_auc']
    ax1.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")

    ap_val = compute_metrics(y_true_v, y_prob_v)['pr_auc']
    ax2.plot(rec, prec, label=f"{name} (AP={ap_val:.4f})")

ax1.plot([0, 1], [0, 1], 'k--')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curves — Top 3 Models (Val)')
ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.4)

ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('PR Curves — Top 3 Models (Val)')
ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_pr_curves.png', dpi=150)
plt.show()
print('Saved → figures/roc_pr_curves.png')

In [ ]:
# Cell 16 — Overfitting analysis: train-val gap per model
gap_data = [
    {
        'Model': name,
        'Train Loss (best epoch)': info['history']['train_loss'][info['best_epoch'] - 1],
        'Val Loss (best epoch)':   info['history']['val_loss'][info['best_epoch'] - 1],
        'Gap':                     round(
            info['history']['val_loss'][info['best_epoch'] - 1]
            - info['history']['train_loss'][info['best_epoch'] - 1],
            6
        ),
        'Epochs run': len(info['history']['train_loss']),
    }
    for name, (_, info) in all_results.items()
]
df_gap = pd.DataFrame(gap_data).sort_values('Gap', ascending=False)
print(df_gap.to_string(index=False))
df_gap.to_csv(RESULTS_DIR / 'overfitting_gap.csv', index=False)

## Observations

**Overfitting pattern:** CTR models overfit fast because high-cardinality embedding tables can memorize training examples after just 1–2 epochs. Early stopping (patience=2) limits this, but the val loss often rises from epoch 2 onward. This is expected behavior — report it, don't hide it.

**Threshold justification:** With a low positive (click) rate, accuracy at 0.5 is misleading — a model that predicts 0 for everything would get high accuracy. We choose the threshold that maximizes F1 **on the validation set** and apply it to the test set, so we never use test labels to make any decision.

**Embedding dim effect:** Larger embedding dim → more parameters → faster overfitting, but potentially higher capacity. The gap analysis above shows how dropout and weight decay interact with this.

**DCN vs MLP:** The cross network explicitly models second-order feature interactions in `O(d)` per cross layer, whereas an MLP needs depth to discover the same interactions implicitly. Whether it helps in practice depends on the data; inspect the loss curves and AUC gap.